# Werewolf Transformer — pilot-002 population continuation (iterations 3–4)

pilot-002 の population research を **iteration 2 から iteration 4 まで**同じ設定のまま再開し、
完了後に直近3 policy / faction を固定して 27 profiles × 20 villages = 540 villages を再評価します。

この notebook では:
1. 既存の `pilot-002/population/population.run.json` を exact resume
2. iteration 3, 4 を追加
3. 学習終了後に新しい fixed evaluation (`fixed-evaluation-last3-v2`) を実施
4. 最後に `PILOT-002 ITERATIONS 3-4 REPORT` を表示

報酬、ルール、action mask、観測、モデル構造、PPO設定は変更しません。
population run-state に保存された既存設定をそのまま使います。

Colab が切れた場合は同じ notebook を上から再実行してください。
population research も fixed evaluation も保存済み地点から不足分だけ継続します。


In [ ]:
# 1) Google Drive を接続
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2) 最新 main を取得して依存関係をインストール
%cd /content
!rm -rf Are-you-werewolf
!git clone --depth 1 https://github.com/dolphin23-jp/Are-you-werewolf.git
%cd /content/Are-you-werewolf/backend
!python -m pip install -q -e ".[rl,transformer]"


In [ ]:
# 3) GPU / pilot-002 の状態確認
from pathlib import Path
import json
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU が有効ではありません。Colab の「ランタイム → ランタイムのタイプを変更」で GPU を選んでください。'
    )

ROOT = Path('/content/drive/MyDrive/werewolf-training/pilot-002')
POOL = ROOT / 'pool'
POPULATION = ROOT / 'population'
POP_STATE = POPULATION / 'population.run.json'
EVAL = ROOT / 'fixed-evaluation-last3-v2'

if not POP_STATE.exists():
    raise RuntimeError(
        f'population run-state がありません: {POP_STATE}\n'
        'pilot-002 iterations 1-2 が完了した同じ Google Drive を使用してください。'
    )
if not (POOL / 'manifest.json').exists():
    raise RuntimeError(f'policy pool がありません: {POOL}')

state = json.loads(POP_STATE.read_text())
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
print('pilot-002:', ROOT)
print('population state:', POP_STATE)
print('saved completed_iterations:', state.get('completed_iterations'))
print('fixed evaluation v2:', EVAL)


In [ ]:
# 4) population research を iteration 4 まで exact resume
#    既存 population.run.json に保存された設定が authoritative です。
import subprocess

events_log = POPULATION / 'events-iterations-3-4.log'
cmd = [
    'python', 'scripts/run_population_iterations_torch.py',
    '--pool-dir', str(POOL),
    '--run-dir', str(POPULATION),
    '--iterations', '4',
    '--resume',
    '--device', 'auto',
]

with events_log.open('a', encoding='utf-8') as log:
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    rc = process.wait()

if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)

print('\nPOPULATION ITERATION 4 TARGET REACHED')


In [ ]:
# 5) iteration 3 / 4 summary の存在確認
for iteration in (3, 4):
    summary = POPULATION / f'iteration-{iteration:04d}' / 'summary.json'
    if not summary.exists():
        raise RuntimeError(f'iteration {iteration} summary がありません: {summary}')
    payload = json.loads(summary.read_text())
    print(
        f"iteration={iteration} "
        f"max_deviation={payload['max_restricted_deviation_gain']:.4f} "
        f"oracles={','.join(payload['oracle_policy_ids'])} "
        f"pool_generation_after={payload['pool_generation_after']}"
    )


In [ ]:
# 6) 学習完了後の直近3 policy/faction を 20 games/profile で固定評価
#    27 profiles × 20 = 540 villages。学習更新は一切しません。
EVAL.mkdir(parents=True, exist_ok=True)
table = EVAL / 'payoffs.json'
measure_log = EVAL / 'measure.log'

cmd = [
    'python', 'scripts/measure_population_payoffs_torch.py',
    '--pool-dir', str(POOL),
    '--table', str(table),
    '--last', '3',
    '--games-per-profile', '20',
    '--extra-games', '0',
    '--seed', '4101',
    '--parallel-games', '16',
    '--inference-batch-size', '64',
    '--device', 'auto',
]

result = subprocess.run(cmd, check=True, text=True, capture_output=True)
print(result.stdout)
measure_log.write_text(result.stdout, encoding='utf-8')


In [ ]:
# 7) fixed evaluation から fresh meta strategy を解く
meta = EVAL / 'meta.json'
meta_log = EVAL / 'meta.log'

cmd = [
    'python', 'scripts/solve_population_meta_torch.py',
    '--table', str(table),
    '--pool-dir', str(POOL),
    '--output', str(meta),
    '--last', '3',
    '--temperature', '0.25',
    '--iterations', '100',
    '--damping', '0.5',
    '--device', 'auto',
]

result = subprocess.run(cmd, check=True, text=True, capture_output=True)
print(result.stdout)
meta_log.write_text(result.stdout, encoding='utf-8')


In [ ]:
# 8) ChatGPT に送る report を表示
from app.engine.roles import Team
from app.training.torch_pool import TorchPolicyPool

pool = TorchPolicyPool(POOL, device='cpu')
targets = {
    team.value: list(pool.policy_ids_for_team(team, last=3))
    for team in Team
}

print('\n===== PILOT-002 ITERATIONS 3-4 REPORT =====')
print('targets:', json.dumps(targets, ensure_ascii=False))

for iteration in (3, 4):
    summary_path = POPULATION / f'iteration-{iteration:04d}' / 'summary.json'
    print(f'\n===== POPULATION SUMMARY {iteration} =====')
    print(json.dumps(json.loads(summary_path.read_text()), ensure_ascii=False, indent=2))
    print(f'===== END POPULATION SUMMARY {iteration} =====')

print('\n===== FIXED EVALUATION META =====')
print(meta_log.read_text().strip())
print('===== END FIXED EVALUATION META =====')

print('\n===== FIXED EVALUATION PROFILE PAYOFFS =====')
print(measure_log.read_text().strip())
print('===== END FIXED EVALUATION PROFILE PAYOFFS =====')

print('\n===== LEARNING EVENTS ITERATIONS 3-4 (LAST 80) =====')
if events_log.exists():
    lines = [line for line in events_log.read_text().splitlines() if line.strip()]
    for line in lines[-80:]:
        print(line)
else:
    print('MISSING events log')
print('===== END LEARNING EVENTS =====')

print('\n===== END PILOT-002 ITERATIONS 3-4 REPORT =====')
print('\nこの report 全体を ChatGPT に貼り付けてください。')


## 途中で Colab が切れた場合

同じ notebook を上から再実行してください。

- population iteration は `population.run.json` から exact resume
- fixed evaluation は `payoffs.json` に既に入っている profile/game を再利用し、不足分だけ測定

学習済み checkpoint を削除したり、Google Drive の `pilot-002` を手動編集したりしないでください。
